# Translate& Summarize News Article Using Open Source Models - Ollama locally 

In this notebook i experimented ollama by downloading and running it in my own PC and checked on how it translates and summarizes the scrapped chinese website text while utilizing a small model of it.

##### First, I checked whether Ollama is runnig locally in my PC

In [ ]:
requests.get("http://localhost:11434").content

##### Then I downloaded the **Llama 3.2** model, which is available in 1B and 3B parameter variants. It is optimized for multilingual dialogue, agentic retrieval, and summarization tasks. More information is available in the [Ollama Llama 3.2 documentation](https://ollama.com/library/llama3.2).

In [ ]:
!ollama pull llama3.2

In [ ]:
# Reusing the article text scraped in Day 1.

from pathlib import Path
import importlib
import sys
import urllib.request
import json

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.services.scraper as scraper_module
scraper_module = importlib.reload(scraper_module)
from src.services.scraper import scrape_article

url = "https://www.bbc.com/zhongwen/articles/cjwgxdz5n49o/trad"

if "article" in locals() and article:
    article_text = article
else:
    article_text = scrape_article(url)

print(article_text[:2000])

In [ ]:
# Translation with Ollama

def ollama_generate(prompt: str, model: str = "llama3.2", system: str | None = None) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }
    if system:
        payload["system"] = system
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as response:
        result = json.loads(response.read().decode("utf-8"))
    return result.get("response", "")

system_prompt = (
    "You are a professional translator. Translate Chinese text to English only. "
    "Do not explain, do not add commentary, and do not leave any Chinese text in the output."
)

translation_prompt = (
    "Translate the following Chinese article into natural English. "
    "Preserve the meaning and key facts. Return only the translated English text.\n\n"
    f"{article_text[:12000]}"
)

translation = ollama_generate(translation_prompt, system=system_prompt)
print(translation[:4000])

In [ ]:
# Summarize the translated article with Ollama
summary_prompt = (
    "Summarize the following article in 3 bullet points. Keep it concise and clear.\n\n"
    f"{translation if 'translation' in locals() and translation else ''}"
)

if 'translation' not in locals() or not translation:
    translation_prompt = (
        "Translate the following article into natural English. Keep the meaning and key facts intact.\n\n"
        f"{article_text[:12000]}"
    )
    translation = ollama_generate(translation_prompt)

summary = ollama_generate(summary_prompt)
print(summary)

##### Here I experimented with the ***DeepSeek-R1:1.5B*** open-source reasoning model. This version is a distilled variant built on Alibaba Cloud's **Qwen** architecture, offering strong reasoning capabilities while remaining lightweight. More information is available on the [Ollama DeepSeek-R1 model page](https://ollama.com/library/deepseek-r1).

In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
# Translation with Ollama-DeepSeek

def ollama_generate(prompt: str, model: str = "deepseek-r1:1.5b", system: str | None = None) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }
    if system:
        payload["system"] = system
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as response:
        result = json.loads(response.read().decode("utf-8"))
    return result.get("response", "")

system_prompt = (
    "You are a professional translator. Translate Chinese text to English only. "
    "Do not explain, do not add commentary, and do not leave any Chinese text in the output."
)

translation_prompt = (
    "Translate the following Chinese article into natural English. "
    "Preserve the meaning and key facts. Return only the translated English text.\n\n"
    f"{article_text[:12000]}"
)

translation = ollama_generate(translation_prompt, system=system_prompt)
print(translation[:4000])

In [ ]:
# Summarize the translated article with Ollama-DeepSeek
summary_prompt = (
    "Summarize the following article in 3 bullet points. Keep it concise and clear.\n\n"
    f"{translation if 'translation' in locals() and translation else ''}"
)

if 'translation' not in locals() or not translation:
    translation_prompt = (
        "Translate the following article into natural English. Keep the meaning and key facts intact.\n\n"
        f"{article_text[:12000]}"
    )
    translation = ollama_generate(translation_prompt)

summary = ollama_generate(summary_prompt)
print(summary)

### Model Comparison: Chinese → English Translation

| Model | Response Time | Translation Quality | Key Observations |
|:------|--------------:|:-------------------:|------------------|
| **Grok Llama 3.3 70B** | **5.4 s** | ⭐⭐⭐⭐⭐ | Most accurate and complete translation. Preserved names, technical terms, quotations, and context with no noticeable hallucinations. |
| **Llama 3.2** | **3 min 39.4 s** | ⭐⭐⭐☆☆ | Produced a reasonable summary but omitted significant portions of the article and contained several translation errors (e.g., incorrect place names). |
| **DeepSeek-R1:1.5B** | **1 min 28.4 s** | ⭐⭐☆☆☆ | Faster than Llama 3.2 but produced numerous mistranslations and hallucinations (e.g., "machine people"), making it unreliable for translation tasks. |


### Key Takeaways

- **Grok Llama 3.3 70B** delivered the **highest translation quality** while also being the **fastest** model.
- **DeepSeek-R1:1.5B** responded faster than Llama 3.2 but sacrificed translation accuracy due to hallucinations and literal translations.
- **Llama 3.2** was the **slowest** model and tended to summarize rather than faithfully translate the source text.
- 📌 **Overall ranking (quality):** Grok Llama 3.3 70B > Llama 3.2 > DeepSeek-R1:1.5B.
- 📌 **Overall ranking (latency):** Grok Llama 3.3 70B > DeepSeek-R1:1.5B > Llama 3.2.